Consider a GPT-2 XL-sized model using our assignment architecture, which has the following
configuration:
vocab_size: 50,257
context_length: 1,024
num_layers: 48
d_model: 1,600
num_heads: 25
d_ff: 4,288 (the nearest multiple of 64 to 8
3 × 1, 600)
Suppose we constructed our model using this configuration. How many trainable parameters
would our model have? Assuming each parameter is represented using single-precision floating
point, how much memory is required to just load this model?
Deliverable: A one-to-two sentence response

1.Each Block:

Token_Embeddings: 50,257 x 1,600 = 80411200

Each layers (48):
q_proj: 1,600 x 1,600 = 2560000
k_proj: 1,600 x 1,600 = 2560000
v_proj: 1,600 x 1,600 = 2560000
output_proj: 1,600 x 1,600 = 2560000
RMSNorm1: 1600 
w1, w2, w3: 1600 x 4288 * 3 = 20582400
rmsnorm2: 1600


RmsNorm_again: 1600
output_embedding: 50257 x 1600 


In [4]:
res = (2560000 * 4 + 3200 + 20582400) * 48 + 80411200* 2 + 1600
res

1640452800

1640452800 would be the number of parameters, 4 bytes each. 1640452800* 4 = 6561811200 -> 6.561 GB? 

Identify the matrix multiplies required to complete a forward pass of our GPT-2 XL-shaped
model. How many FLOPs do these matrix multiplies require in total? Assume that our input
sequence has context_length tokens.

In [10]:
def flops_by_matrix(m, n, p):
    return 2* n*m*p

In [20]:
def calculate_flops(
      vocab_size=50257,
      context_length=1024,
      num_layers=48,
      d_model=1600,
      num_heads=25,
      d_ff=4288,
  ):
  d_k = d_v = d_model // num_heads

  # Token embedding + reverse embedding
  flops_embedding = 2 * flops_by_matrix(d_model, vocab_size, context_length)

  # --- Per layer ---

  # QKV projections
  flops_qkv_proj = 3 * num_heads * flops_by_matrix(d_v, d_model, context_length)

  # Q·K and (QK)·V attention
  flops_qk = num_heads * flops_by_matrix(context_length, d_k, context_length)
  flops_qkv = num_heads * flops_by_matrix(context_length, context_length, d_k)
  flops_qkv_mult = flops_qk + flops_qkv

  # Output projection (combining heads)
  flops_combining_heads = flops_by_matrix(context_length, d_model, d_model)

  # SwiGLU FFN: gate projection + up projection + down projection
  flops_swiglu = (
      flops_by_matrix(d_ff, d_model, context_length)
      + flops_by_matrix(d_model, d_ff, context_length)
      + flops_by_matrix(d_model, d_ff, context_length)
  )

  flops_per_layer = flops_qkv_proj + flops_qkv_mult + flops_combining_heads + flops_swiglu
  flops_all_layers = num_layers * flops_per_layer

  total = flops_embedding + flops_all_layers

  breakdown = {
      "embedding (fwd + rev)": flops_embedding,
      "qkv_proj (per layer)": flops_qkv_proj,
      "qkv_mult (per layer)": flops_qkv_mult,
      "combining_heads (per layer)": flops_combining_heads,
      "swiglu (per layer)": flops_swiglu,
  }

  print(f"Total FLOPs: {total:,.0f} ({total:.2e})")
  print(f"\n{'Component':<30} {'FLOPs':>15} {'% of Total':>10}")
  print("-" * 57)

  for name, flops in breakdown.items():
      scaled = flops * num_layers if "per layer" in name else flops
      print(f"{name:<30} {scaled:>15,.0f} {scaled / total * 100:>9.2f}%")

  return total, breakdown


calculate_flops()

Total FLOPs: 3,681,452,032,000 (3.68e+12)

Component                                FLOPs % of Total
---------------------------------------------------------
embedding (fwd + rev)          329,364,275,200      8.95%
qkv_proj (per layer)           754,974,720,000     20.51%
qkv_mult (per layer)           322,122,547,200      8.75%
combining_heads (per layer)    251,658,240,000      6.84%
swiglu (per layer)             2,023,332,249,600     54.96%


(3681452032000,
 {'embedding (fwd + rev)': 329364275200,
  'qkv_proj (per layer)': 15728640000,
  'qkv_mult (per layer)': 6710886400,
  'combining_heads (per layer)': 5242880000,
  'swiglu (per layer)': 42152755200})

In [ ]:
Based on your analysis above, which parts of the model require the most FLOPs?
Deliverable: A one-to-two sentence response.



 

Repeat your analysis with GPT-2 small (12 layers, 768 d_model, 12 heads), GPT-2 medium
(24 layers, 1024 d_model, 16 heads), and GPT-2 large (36 layers, 1280 d_model, 20 heads). As
the model size increases, which parts of the Transformer LM take up proportionally more or
less of the total FLOPs?

In [23]:
calculate_flops(vocab_size=50257, context_length=1024, num_layers=12, d_model=768,  num_heads=12, d_ff=3072)
calculate_flops(vocab_size=50257, context_length=1024, num_layers=24, d_model=1024, num_heads=16, d_ff=4096)
calculate_flops(vocab_size=50257, context_length=1024, num_layers=36, d_model=1280, num_heads=20, d_ff=5120)

Total FLOPs: 428,677,791,744 (4.29e+11)

Component                                FLOPs % of Total
---------------------------------------------------------
embedding (fwd + rev)          158,094,852,096     36.88%
qkv_proj (per layer)            43,486,543,872     10.14%
qkv_mult (per layer)            38,654,705,664      9.02%
combining_heads (per layer)     14,495,514,624      3.38%
swiglu (per layer)             173,946,175,488     40.58%
Total FLOPs: 1,138,506,072,064 (1.14e+12)

Component                                FLOPs % of Total
---------------------------------------------------------
embedding (fwd + rev)          210,793,136,128     18.51%
qkv_proj (per layer)           154,618,822,656     13.58%
qkv_mult (per layer)           103,079,215,104      9.05%
combining_heads (per layer)     51,539,607,552      4.53%
swiglu (per layer)             618,475,290,624     54.32%
Total FLOPs: 2,389,500,231,680 (2.39e+12)

Component                                FLOPs % of Total
---

(2389500231680,
 {'embedding (fwd + rev)': 263491420160,
  'qkv_proj (per layer)': 10066329600,
  'qkv_mult (per layer)': 5368709120,
  'combining_heads (per layer)': 3355443200,
  'swiglu (per layer)': 40265318400})

As model grows, ffn takes up more. If context length grow, because attention score is proportion to the squre of context_length, it gorws a lot.
If d_model grows, combining heads and ffn grows to d_model^2. FFn grows more since it's almost like >6x depending on d_ff.


Take GPT-2 XL and increase the context length to 16,384. How does the total FLOPs for one
forward pass change? How does the relative contribution of FLOPs of the model components
change?

In [22]:
calculate_flops(context_length = 16384)

Total FLOPs: 136,212,643,840,000 (1.36e+14)

Component                                FLOPs % of Total
---------------------------------------------------------
embedding (fwd + rev)          5,269,828,403,200      3.87%
qkv_proj (per layer)           12,079,595,520,000      8.87%
qkv_mult (per layer)           82,463,372,083,200     60.54%
combining_heads (per layer)    4,026,531,840,000      2.96%
swiglu (per layer)             32,373,315,993,600     23.77%


(136212643840000,
 {'embedding (fwd + rev)': 5269828403200,
  'qkv_proj (per layer)': 251658240000,
  'qkv_mult (per layer)': 1717986918400,
  'combining_heads (per layer)': 83886080000,
  'swiglu (per layer)': 674444083200})

QKV consumes more

Let us compute how much memory and compute running AdamW requires. Assume we are using
float32 for every tensor.
(a) How much peak memory does running AdamW require? Decompose your answer based on the
memory usage of the parameters, activations, gradients, and optimizer state. Express your
answer in terms of the batch_size and the model hyperparameters (vocab_size,
context_length, num_layers, d_model, num_heads). Assume 𝑑ff =
8
3 × 𝑑model.
For simplicity, when calculating memory usage of activations, consider only the following
components:
• Transformer block
‣ RMSNorm(s)
‣ Multi-head self-attention sublayer: 𝑄𝐾𝑉 projections, 𝑄𝐾⊤ matrix multiply, softmax,
weighted sum of values, output projection.
‣ Position-wise feed-forward (SwiGLU): 𝑊 1, 𝑊 2, SiLU on the gate branch, element-wise
product, 𝑊 3
32
• final RMSNorm
• output embedding
• cross-entropy on logits
Deliverable: An algebraic expression

# graident always = param, optimizer_state_extra always = 2 * param 
# so total_bytes = 4 * param + activation 

 Parameters (P)

  ┌────────────────────────────────────┬──────────┐
  │             Component              │ Elements │
  ├────────────────────────────────────┼──────────┤
  │ 2 RMSNorms × L                     │ 2Ld      │
  ├────────────────────────────────────┼──────────┤
  │ Attention (W_Q, W_K, W_V, W_O) × L │ 4Ld²     │
  ├────────────────────────────────────┼──────────┤
  │ SwiGLU (W1, W2, W3) × L            │ 8Ld²     │
  ├────────────────────────────────────┼──────────┤
  │ Final RMSNorm                      │ d        │
  ├────────────────────────────────────┼──────────┤
  │ Output embedding                   │ Vd       │
  └────────────────────────────────────┴──────────┘

  P = L(12d² + 2d) + d + Vd

  Gradients

  G = P

  Optimizer State

  O = 2P (AdamW m + v, already fp32 so no master copy)

  Activations (A)

  Per transformer block:

  ┌───────────────────┬─────────────────────────────┬──────────┐
  │     Operation     │        What's saved         │ Elements │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ RMSNorm1          │ input                       │ BTd      │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ RMSNorm2          │ input                       │ BTd      │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ QKV projections   │ input + Q + K + V           │ 4BTd     │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ Softmax           │ output                      │ BHT²     │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ Weighted sum (AV) │ output                      │ BTd      │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ W1 linear         │ input (shared w/ W2)        │ BTd      │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ SiLU              │ input xW1                   │ 8BTd/3   │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ Elem-wise product │ both inputs: SiLU(xW1), xW2 │ 16BTd/3  │
  ├───────────────────┼─────────────────────────────┼──────────┤
  │ W3 linear         │ input (gate ⊙ h2)           │ 8BTd/3   │
  └───────────────────┴─────────────────────────────┴──────────┘

  Per block = 8BTd + BHT² + 32BTd/3 = BT(56d/3 + HT)

  Other layers:

  ┌───────────────────────────────┬──────────┐
  │           Component           │ Elements │
  ├───────────────────────────────┼──────────┤
  │ Final RMSNorm input           │ BTd      │
  ├───────────────────────────────┼──────────┤
  │ Output embedding input        │ BTd      │
  ├───────────────────────────────┼──────────┤
  │ Cross-entropy (softmax probs) │ BTV      │
  └───────────────────────────────┴──────────┘

  A = LBT(56d/3 + HT) + 2BTd + BTV

  Total

  In elements:

  ▎ Total = 4P + A
  ▎ = 4[L(12d² + 2d) + d + Vd] + LBT(56d/3 + HT) + 2BTd + BTV

  In bytes (×4 for fp32):

  ▎ Total bytes = 16P + 4A


  



In [24]:
def calculate_memory_usage(batch_size, vocab_size, context_length, num_layers, d_model, num_heads):
    B = batch_size
    T = context_length
    d = d_model
    H = num_heads
    L = num_layers
    V = vocab_size
    d_ff = 8 * d // 3

    # Parameters
    params = L * (12 * d**2 + 2 * d) + d + V * d

    # Gradients = Parameters
    gradients = params

    # Optimizer state: AdamW fp32 (m + v)
    optimizer = 2 * params

    # Activations (no recomputation)
    # Per block: 8BTd + BHT^2 + 32BTd/3
    per_block = 8 * B * T * d + B * H * T**2 + 4 * B * T * d_ff
    activations = L * per_block + 2 * B * T * d + B * T * V

    total_elements = params + gradients + optimizer + activations
    total_bytes = total_elements * 4  # fp32

    return {
        "parameters": params,
        "gradients": gradients,
        "optimizer_state": optimizer,
        "activations": activations,
        "total_elements": total_elements,
        "total_bytes": total_bytes,
        "total_gb": total_bytes / (1024**3),
    }

Total bytes = 16L(12d² + 2d) + 16d(1 + V) + 4BT(L(56d/3 + HT) + 2d + V)

In [25]:
print(calculate_memory_usage(5, 50257, 1024, 48, 1600, 25))


{'parameters': 1555126400, 'gradients': 1555126400, 'optimizer_state': 3110252800, 'activations': 13904532480, 'total_elements': 20125038080, 'total_bytes': 80500152320, 'total_gb': 74.97160911560059}


In [ ]:
Instantiate your answer for a GPT-2 XL-shaped model to get an expression that only
depends on the batch_size. What is the maximum batch size you can use and still fit within
80GB memory?
Deliverable: An expression that looks like 𝑎 ⋅ b







In [30]:
# This is the constant term
L = 48
d = 1600
V = 50257
b = 16 * L * (12 * (d**2) + 2 * d) + 16 * d * (1+ V)
H = 25
T = 1024 
print(b)

24882022400


In [32]:
a =  4 * T * (L * (56 * d / 3 + H * T) + 2 * d + V)
print(a)

11124150272.0


5 batches in 80 G. a, b as above. 

Per parameter, AdamW performs these operations:
     
  m  = β₁·m + (1-β₁)·g           # 2 mul + 1 add  =  3
  v  = β₂·v + (1-β₂)·g²          # 3 mul + 1 add  =  4
  m̂  = m / (1-β₁ᵗ)               # 1 mul           =  1
  v̂  = v / (1-β₂ᵗ)               # 1 mul           =  1
  θ  = θ - lr·(m̂/(√v̂+ε) + λ·θ)  # 1√ + 1÷ + 2× + 3±  =  7
                                   # ─────────────────────
                                   # Total: 16 per param


16 * P = 16 * L * (12 * (d**2) + 2 * d) + 16 * d * (1+ V)

Model FLOPs utilization (MFU) is defined as the ratio of observed throughput (tokens per
second) relative to the hardware’s theoretical peak FLOP throughput
[A. Chowdhery et al., 2022]. An NVIDIA H100 GPU has a theoretical peak of 495 teraFLOP/
s for “float32” (actually TensorFloat-32, which in reality is “bfloat19”) operations. Assuming
you are able to get 50% MFU, how long would it take to train a GPT-2 XL for 400K steps
and a batch size of 1024 on a single H100? Following J. Kaplan et al. [25] and
J. Hoffmann et al. [26], assume that the backward pass has twice the FLOPs of the forward
pass.
Deliverable: The number of hours training would take, with a brief justification.

In [33]:
print(calculate_memory_usage(1024, 50257, 1024, 48, 1600, 25))

{'parameters': 1555126400, 'gradients': 1555126400, 'optimizer_state': 3110252800, 'activations': 2847648251904, 'total_elements': 2853868757504, 'total_bytes': 11415475030016, 'total_gb': 10631.489595413208}


In [41]:
flops = 3681452032000 * 3 * 400 * 1000 * 1024

# 3681452032000 is from above when calcuating flops. 3 because 1 forward + 2 * backward

In [44]:
time = flops / (495 /2 * 1e12)

In [45]:
time/3600

5077.180984199326